---
# PARTIE 2 -- L'API DataFrame

## 2.1 Du RDD au DataFrame

L'API RDD est puissante mais verbeuse et peu optimisee : Spark ne sait pas ce que
contiennent les elements (ils sont des `object` Python opaques). L'API **DataFrame**
introduit un schema explicite, ce qui permet a l'optimiseur Catalyst de generer
un plan d'execution bien plus efficace.

### Creation depuis un RDD


In [19]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType, TimestampType
)

# Schema explicite -- toujours preferable a l'inference automatique
schema_velib = StructType([
    StructField("station_id",       IntegerType(), nullable=False),
    StructField("nom_station",      StringType(),  nullable=True),
    StructField("code_arr",         IntegerType(), nullable=True),
    StructField("capacite",         IntegerType(), nullable=True),
    StructField("velos_meca",       IntegerType(), nullable=True),
    StructField("velos_elec",       IntegerType(), nullable=True),
    StructField("bornettes_libres", IntegerType(), nullable=True),
    StructField("horodatage",       StringType(),  nullable=True),
])

# Convertir le RDD de dicts en RDD de Row, puis en DataFrame.
# ADAPTATION : le modèle partait de `step3`, mais `step3` ne contient que les relevés à
# taux < 10 % ET porte deux clés en plus (nb_velos, taux_occupation) : createDataFrame
# échouerait (10 champs pour un schéma de 8) et le DataFrame serait un échantillon
# biaisé du réseau. On part donc de `valid_rdd` : tous les relevés, exactement 8 champs.
row_rdd = valid_rdd.map(lambda d: Row(**d))
df_from_rdd = spark.createDataFrame(row_rdd, schema=schema_velib)

df_from_rdd.printSchema()
df_from_rdd.show(5, truncate=False)
print(f"Lignes : {df_from_rdd.count():,}  --  Partitions : {df_from_rdd.rdd.getNumPartitions()}")

root
 |-- station_id: integer (nullable = false)
 |-- nom_station: string (nullable = true)
 |-- code_arr: integer (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- horodatage: string (nullable = true)



+----------+-----------------------------------+--------+--------+----------+----------+----------------+-------------------------+
|station_id|nom_station                        |code_arr|capacite|velos_meca|velos_elec|bornettes_libres|horodatage               |
+----------+-----------------------------------+--------+--------+----------+----------+----------------+-------------------------+
|1098      |Benjamin Godard - Victor Hugo      |16      |35      |4         |5         |26              |2020-11-26T12:59:00+00:00|
|1031      |André Mazet - Saint-André des Arts |6       |55      |23        |4         |28              |2020-11-26T12:59:00+00:00|
|1225      |Charonne - Robert et Sonia Delauney|11      |20      |0         |0         |20              |2020-11-26T12:59:00+00:00|
|2330      |Toudouze - Clauzel                 |9       |21      |0         |1         |20              |2020-11-26T12:59:00+00:00|
|1747      |Mairie du 12ème                    |12      |30      |3         

Lignes : 10,769,264  --  Partitions : 10


---
## 2.2 Lecture directe depuis les fichiers Parquet

En pratique, on ne passe presque jamais par les RDD pour creer un DataFrame.
Spark peut lire directement de nombreux formats (CSV, JSON, Parquet, Delta, ORC...).

Le format **Parquet** est le format de facto pour les donnees analytiques :
- Stockage colonnaire (lecture partielle possible)
- Compression integree (Snappy, Zstandard...)
- Schema embarque dans les fichiers
- Partitionnement natif (Hive-style)


### Préparation du Parquet « pré-préparé »

L'énoncé suppose un extrait Parquet déjà partitionné, distribué aux étudiants. Il n'est pas
disponible : on le fabrique ici **avec Spark**. Deux enseignements au passage :

- `spark.read.csv(...)` (lecteur natif de la JVM) évite de faire transiter chaque ligne par
  Python, contrairement au chemin RDD -> `Row` -> DataFrame de la cellule précédente
  (comparez les durées dans le Spark UI).
- `partitionBy("annee", "mois")` crée des dossiers `annee=2021/mois=1/` : lors d'une lecture
  filtrée sur ces colonnes, Spark n'ouvre que les dossiers concernés (*partition pruning*).

In [20]:
from pyspark.sql import functions as F

if not any(VELIB_PARQ_DIR.glob("annee=*")):
    t0 = time.perf_counter()
    (
        spark.read
        .option("sep", ";")
        .option("header", True)      # ignore l'en-tête de CHAQUE fichier .csv.gz
        .schema(schema_velib)        # schéma explicite : pas de 2e passe d'inférence
        .csv(str(VELIB_RAW_DIR / "*.csv.gz"))
        # annee / mois extraits du texte ISO "AAAA-MM-JJT..." : plus simple et plus sûr qu'un
        # parsing de timestamp, et sans dépendance au fuseau de la session.
        .withColumn("annee", F.substring("horodatage", 1, 4).cast("int"))
        .withColumn("mois",  F.substring("horodatage", 6, 2).cast("int"))
        .write.mode("overwrite")
        .partitionBy("annee", "mois")
        .parquet(str(VELIB_PARQ_DIR))
    )
    print(f"Parquet écrit dans {VELIB_PARQ_DIR} en {time.perf_counter() - t0:.1f} s")
else:
    print(f"Parquet déjà présent : {VELIB_PARQ_DIR}")

Parquet écrit dans ../data/velib/parquet en 5.0 s


In [21]:
# Lecture des fichiers Parquet pre-prepares (partitionnes par annee et mois)
# Spark detecte automatiquement le schema et les colonnes de partition
# Spark lit le schéma dans le pied de page des fichiers Parquet (aucune inférence coûteuse) et
# reconstitue annee / mois à partir des noms de dossiers (annee=2021/mois=1).
df_velib = spark.read.parquet(str(VELIB_PARQ_DIR))

df_velib.printSchema()
print(f"\nDimensions : {df_velib.count():,} lignes x {len(df_velib.columns)} colonnes")
print(f"Partitions : {df_velib.rdd.getNumPartitions()}")

root
 |-- station_id: integer (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- code_arr: integer (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- horodatage: string (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)




Dimensions : 10,769,264 lignes x 10 colonnes


Partitions : 17


In [22]:
# Lecture selective d'une partition (predicate pushdown)
# Spark ne lira que les fichiers du dossier annee=2021/mois=02
# On filtre sur les colonnes de partition : Spark supprime les autres dossiers du plan
# AVANT de lire quoi que ce soit (« partition pruning »).
df_fev_2021 = df_velib.filter("annee = 2021 AND mois = 2")
print(f"Février 2021 : {df_fev_2021.count():,} lignes")

# Comparer avec une lecture complete puis filtre -- le plan d'execution est identique
# grace au predicate pushdown, mais on le verifie avec explain()
print("\n--- Plan d'execution (filtre apres lecture) ---")
df_velib.filter("annee = 2021 AND mois = 2").explain(mode="formatted")
# À repérer dans le plan : « PartitionFilters: [annee = 2021, mois = 2] » sur le nœud Scan parquet.

Février 2021 : 2,182,770 lignes

--- Plan d'execution (filtre apres lecture) ---
== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [10]: [station_id#118, nom_station#119, code_arr#120, capacite#121, velos_meca#122, velos_elec#123, bornettes_libres#124, horodatage#125, annee#126, mois#127]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/data/velib/parquet]
PartitionFilters: [isnotnull(annee#126), isnotnull(mois#127), (annee#126 = 2021), (mois#127 = 2)]
ReadSchema: struct<station_id:int,nom_station:string,code_arr:int,capacite:int,velos_meca:int,velos_elec:int,bornettes_libres:int,horodatage:string>

(2) ColumnarToRow [codegen id : 1]
Input [10]: [station_id#118, nom_station#119, code_arr#120, capacite#121, velos_meca#122, velos_elec#123, bornettes_libres#124, horodatage#125, annee#126, mois#127]




---
## 2.3 Exploration et diagnostic des donnees

Avant tout traitement, il faut comprendre la structure et la qualite des donnees.
Spark propose des fonctions d'agregation statistiques integrees.


In [23]:
from pyspark.sql import functions as F

# Vue d'ensemble statistique
print("=== Statistiques descriptives (colonnes numeriques) ===")
# summary() est une version étendue de describe() : elle ajoute les quartiles.
# Un min négatif ou une capacité minimale de 0 signalent d'emblée des anomalies.
df_velib.select("capacite", "velos_meca", "velos_elec", "bornettes_libres").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).show()

=== Statistiques descriptives (colonnes numeriques) ===


+-------+------------------+-----------------+------------------+------------------+
|summary|          capacite|       velos_meca|        velos_elec|  bornettes_libres|
+-------+------------------+-----------------+------------------+------------------+
|  count|          10769264|         10769264|          10769264|          10769264|
|   mean|31.601376937179737|8.003176354484392|3.5723758838115587|20.025824698883785|
| stddev|11.743355782587848|9.028600647567174|3.1363441349315937|11.590994637915811|
|    min|                 0|                0|                 0|               -41|
|    25%|                23|                2|                 1|                12|
|    50%|                30|                5|                 3|                19|
|    75%|                37|               11|                 5|                27|
|    max|                74|               73|                41|                71|
+-------+------------------+-----------------+------------------+

In [24]:
# Comptage des valeurs nulles par colonne
print("=== Valeurs nulles par colonne ===")
# count(when(cond, x)) ne compte que les lignes où la condition est vraie (when() renvoie
# NULL sinon, et count ignore les NULL). On obtient ainsi TOUS les comptages en UNE seule
# passe sur les données, au lieu d'un filter().count() par colonne (10 passes).
df_velib.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_velib.columns]
).show(vertical=True)

=== Valeurs nulles par colonne ===


-RECORD 0---------------
 station_id       | 0   
 nom_station      | 0   
 code_arr         | 0   
 capacite         | 0   
 velos_meca       | 0   
 velos_elec       | 0   
 bornettes_libres | 0   
 horodatage       | 0   
 annee            | 0   
 mois             | 0   



In [25]:
# Detection des anomalies evidentes
print("=== Anomalies detectees ===")
# Stations avec capacite <= 0
n_cap_zero = df_velib.filter(F.col("capacite") <= 0).count()
print(f"  Snapshots avec capacite <= 0          : {n_cap_zero:,}")

# Stations avec plus de velos que de capacite
# (je tolère +2 : le comptage des vélos et la capacité datent d'instants proches)
n_debord = df_velib.filter(
    (F.col("velos_meca") + F.col("velos_elec")) > F.col("capacite") + 2
).count()
print(f"  Snapshots avec total > capacite + 2   : {n_debord:,}")

# Valeurs negatives
# J'ai calculé bornettes_libres = capacité - vélos sans borne : la valeur devient négative
# quand une station affiche plus de vélos que de places.
for col_name in ["velos_meca", "velos_elec", "bornettes_libres"]:
    n_neg = df_velib.filter(F.col(col_name) < 0).count()
    print(f"  {col_name:<30} < 0 : {n_neg:,}")

=== Anomalies detectees ===
  Snapshots avec capacite <= 0          : 2


  Snapshots avec total > capacite + 2   : 32
  velos_meca                     < 0 : 0
  velos_elec                     < 0 : 0


  bornettes_libres               < 0 : 2,005


In [26]:
# Distribution temporelle : combien de snapshots par mois ?
print("=== Couverture temporelle ===")
# Sert à vérifier qu'il n'y a pas de « trou » dans la collecte (mois sous-représenté).
# Novembre 2020 et avril 2021 sont partiels : la collecte commence le 26/11 et s'arrête le 09/04.
(
    df_velib
    .groupBy("annee", "mois")
    .agg(
        F.count("*").alias("nb_releves"),
        F.countDistinct("station_id").alias("nb_stations"),
        F.countDistinct("horodatage").alias("nb_instants"),
    )
    .orderBy("annee", "mois")
    .show()
)

=== Couverture temporelle ===


+-----+----+----------+-----------+-----------+
|annee|mois|nb_releves|nb_stations|nb_instants|
+-----+----+----------+-----------+-----------+
| 2020|  11|    537439|       1371|        393|
| 2020|  12|   2763930|       1377|       2011|
| 2021|   1|   1487844|       1377|       1085|
| 2021|   2|   2182770|       1372|       1597|
| 2021|   3|   2800281|       1371|       2050|
| 2021|   4|    997000|       1368|        730|
+-----+----+----------+-----------+-----------+



---
## 2.4 Nettoyage et construction des features

Nous allons maintenant construire la table `disponibilite` propre, telle que
definie dans le schema cible du projet.


In [27]:
from pyspark.sql.functions import (
    to_timestamp, col, when, round as spark_round,
    year, month, dayofweek, hour, expr
)

# ── 1. Parsing et typage du timestamp ───────────────────────────────────────
# "2021-01-01T00:47:00+00:00" (texte) -> TimestampType. Les données étant en UTC, on fixe la
# session en UTC (voir carnet 1) : un instant reste alors le même partout.
df_clean = df_velib.withColumn("horodatage", to_timestamp("horodatage"))

# ── 2. Suppression des lignes avec timestamp invalide ───────────────────────
# to_timestamp renvoie NULL quand le texte est illisible (au lieu de lever une erreur).
df_clean = df_clean.filter(col("horodatage").isNotNull())

# ── 3. Suppression des stations avec capacite invalide ──────────────────────
# capacité <= 0 : le taux d'occupation (division par la capacité) n'aurait aucun sens.
df_clean = df_clean.filter(col("capacite") > 0)

# ── 4. Correction des valeurs negatives (bornage a 0) ───────────────────────
# greatest(x, 0) = max(x, 0), et vaut NULL seulement si toutes ses entrées le sont.
# On CORRIGE plutôt que supprimer : une seule mesure négative ne doit pas faire perdre
# le relevé entier de la station.
for c in ["velos_meca", "velos_elec", "bornettes_libres"]:
    df_clean = df_clean.withColumn(c, F.greatest(col(c), F.lit(0)))

# ── 5. Calcul du taux d'occupation ──────────────────────────────────────────
# Définition du schéma cible : (capacité - bornettes libres) / capacité.
# 0 = station vide (aucun vélo), 1 = station pleine (aucune place libre).
df_clean = df_clean.withColumn(
    "taux_occupation", (col("capacite") - col("bornettes_libres")) / col("capacite")
)

# ── 6. Bornage du taux entre 0 et 1 ─────────────────────────────────────────
# Après l'étape 4 le taux ne peut plus dépasser 1 que si bornettes < 0 (déjà corrigé), mais
# on borne par sécurité : les étapes suivantes (features, ML) supposent un taux dans [0, 1].
df_clean = df_clean.withColumn(
    "taux_occupation",
    spark_round(F.least(F.greatest(col("taux_occupation"), F.lit(0.0)), F.lit(1.0)), 4),
)

# ── 7. Extraction de features temporelles (utiles pour le ML en Jour 3) ─────
# Ajouter les colonnes : année, moois, jour_sem, heure, is_weekend
# Les horodatages sont en UTC ; les comportements des usagers suivent l'heure de PARIS
# (pointe du matin 8 h locale = 7 h UTC en hiver). Les colonnes calendaires sont donc
# calculées sur l'heure locale : c'est ce qui rend « entre 7 h et 10 h » (carnet 3) exact.
horodatage_paris = F.from_utc_timestamp(col("horodatage"), "Europe/Paris")
df_clean = (
    df_clean
    .withColumn("annee",       year(horodatage_paris))
    .withColumn("mois",        month(horodatage_paris))
    .withColumn("jour_sem",    dayofweek(horodatage_paris))    # 1 = dimanche ... 7 = samedi (convention Spark)
    .withColumn("heure",       hour(horodatage_paris))
    .withColumn("est_weekend", col("jour_sem").isin(1, 7))     # samedi ou dimanche
)

print(f"Lignes apres nettoyage : {df_clean.count():,}")
df_clean.printSchema()
df_clean.show(3, truncate=True)

Lignes apres nettoyage : 10,769,262
root
 |-- station_id: integer (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- code_arr: integer (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- velos_meca: integer (nullable = false)
 |-- velos_elec: integer (nullable = false)
 |-- bornettes_libres: integer (nullable = false)
 |-- horodatage: timestamp (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- jour_sem: integer (nullable = true)
 |-- heure: integer (nullable = true)
 |-- est_weekend: boolean (nullable = true)

+----------+--------------------+--------+--------+----------+----------+----------------+-------------------+-----+----+---------------+--------+-----+-----------+
|station_id|         nom_station|code_arr|capacite|velos_meca|velos_elec|bornettes_libres|         horodatage|annee|mois|taux_occupation|jour_sem|heure|est_weekend|
+----------+--------------------

In [28]:
# [EXERCICE]
# Ajoutez une colonne "statut" de type StringType avec les valeurs :
#   "vide"   si taux_occupation < 0.1
#   "plein"  si taux_occupation > 0.9
#   "normal" sinon
#
# Indice : utilisez F.when(...).when(...).otherwise(...)
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :
# Des when() enchaînés valent un CASE WHEN SQL : Spark teste les conditions dans l'ordre et la
# première vraie l'emporte. otherwise() prend tout le reste, dont le cas « normal ».
df_clean = df_clean.withColumn(
    "statut",
    F.when(F.col("taux_occupation") < 0.1, "vide")
     .when(F.col("taux_occupation") > 0.9, "plein")
     .otherwise("normal")
)

df_clean.groupBy("statut").count().orderBy("statut").show()

+------+-------+
|statut|  count|
+------+-------+
|normal|8617867|
| plein| 428469|
|  vide|1722926|
+------+-------+



---
## 2.5 Chargement des donnees meteorologiques

> **Adaptation** : l'enseignant a indiqué (26/06) une nouvelle source : l'API **Open-Meteo**
> (archive horaire, sans clé). Le fichier `paris_montsouris_horaire.csv` produit en 0.1 en est
> issu. Le format diffère du SYNOP initial : pas de Kelvin à convertir, un vent déjà en km/h.

| Colonne Open-Meteo | Signification | Unité |
|--------------------|---------------|-------|
| `time`                | Horodatage (`AAAA-MM-JJTHH:MM`) | UTC |
| `temperature_2m`      | Température à 2 m               | °C |
| `relativehumidity_2m` | Humidité relative               | % |
| `windspeed_10m`       | Vitesse du vent à 10 m          | km/h |
| `precipitation`       | Précipitations sur l'heure      | mm |

In [29]:
# Lecture du CSV avec separateur point-virgule.
# Schéma explicite (bonne pratique du carnet) : le lire sans schéma forcerait une passe
# supplémentaire d'inférence et typerait `time` en texte de toute façon.
schema_meteo = StructType([
    StructField("time",                StringType(), True),
    StructField("temperature_2m",      DoubleType(), True),
    StructField("precipitation",       DoubleType(), True),
    StructField("windspeed_10m",       DoubleType(), True),
    StructField("relativehumidity_2m", DoubleType(), True),
])

df_meteo_raw = (
    spark.read
    .option("sep", ";")
    .option("header", True)
    .schema(schema_meteo)
    .csv(str(METEO_CSV))     # les cases vides sont lues comme NULL par défaut
)

print(f"Lignes meteo brutes : {df_meteo_raw.count():,}")
df_meteo_raw.show(5)

Lignes meteo brutes : 4,344
+----------------+--------------+-------------+-------------+-------------------+
|            time|temperature_2m|precipitation|windspeed_10m|relativehumidity_2m|
+----------------+--------------+-------------+-------------+-------------------+
|2020-11-01T00:00|          13.1|          0.0|         12.5|               87.0|
|2020-11-01T01:00|          13.2|          0.0|         10.2|               84.0|
|2020-11-01T02:00|          13.0|          0.1|          9.5|               87.0|
|2020-11-01T03:00|          13.0|          0.7|          8.8|               93.0|
|2020-11-01T04:00|          13.6|          0.5|         13.5|               92.0|
+----------------+--------------+-------------+-------------+-------------------+
only showing top 5 rows



In [30]:
from pyspark.sql.functions import (
    to_timestamp, substring, concat_ws,
    lit, lpad
)

# Transformation du champ time ("2021-01-01T08:00") en timestamp
# Sélectionner "horodatage_meteo", "temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm"
# Supprimer les lignes vides
# Je donne le format explicitement : « yyyy-MM-dd'T'HH:mm » (le T littéral entre apostrophes).
df_meteo = (
    df_meteo_raw
    .select(
        to_timestamp("time", "yyyy-MM-dd'T'HH:mm").alias("horodatage_meteo"),
        col("temperature_2m").alias("temperature_c"),
        col("relativehumidity_2m").alias("humidite_pct"),
        col("windspeed_10m").alias("vent_kmh"),            # déjà en km/h : aucune conversion
        col("precipitation").alias("precipitation_mm"),
    )
    # « Lignes vides » : pas d'horodatage exploitable, ou aucune des 4 mesures renseignée.
    .dropna(subset=["horodatage_meteo"])
    .dropna(how="all", subset=["temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm"])
)

# Ajout de l'indicateur pluie
# Définition du schéma cible : pluie <=> plus de 0,5 mm sur l'heure. coalesce(..., False)
# évite un NULL (ni pluie ni non-pluie) quand la précipitation est manquante.
df_meteo = df_meteo.withColumn(
    "est_pluie", F.coalesce(col("precipitation_mm") > 0.5, F.lit(False))
)

print(f"Lignes meteo nettoyees : {df_meteo.count():,}")
df_meteo.show(8)
df_meteo.describe(["temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm"]).show()

Lignes meteo nettoyees : 4,344


+-------------------+-------------+------------+--------+----------------+---------+
|   horodatage_meteo|temperature_c|humidite_pct|vent_kmh|precipitation_mm|est_pluie|
+-------------------+-------------+------------+--------+----------------+---------+
|2020-11-01 00:00:00|         13.1|        87.0|    12.5|             0.0|    false|
|2020-11-01 01:00:00|         13.2|        84.0|    10.2|             0.0|    false|
|2020-11-01 02:00:00|         13.0|        87.0|     9.5|             0.1|    false|
|2020-11-01 03:00:00|         13.0|        93.0|     8.8|             0.7|     true|
|2020-11-01 04:00:00|         13.6|        92.0|    13.5|             0.5|    false|
|2020-11-01 05:00:00|         14.4|        93.0|    16.1|             0.0|    false|
|2020-11-01 06:00:00|         15.2|        93.0|    20.7|             0.0|    false|
|2020-11-01 07:00:00|         15.9|        91.0|    24.9|             0.0|    false|
+-------------------+-------------+------------+--------+--------

+-------+-----------------+------------------+-----------------+-------------------+
|summary|    temperature_c|      humidite_pct|         vent_kmh|   precipitation_mm|
+-------+-----------------+------------------+-----------------+-------------------+
|  count|             4344|              4344|             4344|               4344|
|   mean|6.908908839779005| 79.51081952117863| 13.5181169429098|0.08501381215469637|
| stddev|4.823007957987826|16.005003617533532|6.708607304985576|0.30790281650171214|
|    min|             -6.0|              24.0|              0.5|                0.0|
|    max|             23.8|             100.0|             45.2|                4.1|
+-------+-----------------+------------------+-----------------+-------------------+



---
## 2.6 Jointure temporelle : Velib' x Meteo

Les observations Velib' sont a la minute, les donnees SYNOP a l'heure.
Pour joindre les deux tables, on **tronque** les horodatages Velib' a l'heure la plus
proche, puis on effectue une jointure sur cette cle commune.


In [31]:
from pyspark.sql.functions import date_trunc, broadcast

# ── 1. Cle de jointure : heure tronquee ──────────────────────────────────────
# date_trunc("hour", ...) ramène 08:47 à 08:00 : chaque relevé Velib' se rattache ainsi
# à l'observation météo de son heure. La troncature est en UTC des deux côtés (session UTC).
df_velib_join = df_clean.withColumn(
    "heure_tronquee",
    date_trunc("hour", col("horodatage")),
)

df_meteo_join = df_meteo.withColumnRenamed(
    "horodatage_meteo", "heure_tronquee"
)

# ── 2. Broadcast join ─────────────────────────────────────────────────────────
# La table meteo est petite (~4 300 lignes pour ces 5 mois d'observations horaires).
# Avec broadcast(), Spark envoie une copie complete a chaque executeur,
# ce qui evite entierement le shuffle de la grosse table Velib'.
# Regles empiriques : utiliser broadcast() si la petite table < quelques centaines de MB.

# how="left" : on conserve TOUS les relevés Velib', même sans météo correspondante (les
# colonnes météo sont alors NULL) ; un inner join ferait disparaître silencieusement des relevés.
df_joint = df_velib_join.join(broadcast(df_meteo_join), on="heure_tronquee", how="left")

# Verification
n_avant  = df_clean.count()
n_apres  = df_joint.count()
n_meteo_null = df_joint.filter(col("temperature_c").isNull()).count()

print(f"Lignes avant jointure  : {n_avant:,}")
print(f"Lignes apres jointure  : {n_apres:,}")
print(f"Snapshots sans meteo   : {n_meteo_null:,} ({100*n_meteo_null/n_apres:.1f}%)")
# n_avant == n_apres prouve que la jointure n'a ni perdu ni dupliqué de relevés
# (la table météo a bien une seule ligne par heure).
assert n_avant == n_apres, "La jointure a modifié le nombre de lignes : doublons météo ?"

Lignes avant jointure  : 10,769,262
Lignes apres jointure  : 10,769,262
Snapshots sans meteo   : 0 (0.0%)


In [32]:
# Examinons le plan d'execution pour verifier que le broadcast est bien utilise
print("=== Plan d'execution de la jointure ===")
df_joint.explain(mode="formatted")
# Cherchez "BroadcastHashJoin" dans le plan -- pas de "SortMergeJoin" (qui implique un shuffle)


=== Plan d'execution de la jointure ===
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- BroadcastHashJoin LeftOuter BuildRight (10)
      :- Project (5)
      :  +- Project (4)
      :     +- Project (3)
      :        +- Filter (2)
      :           +- Scan parquet  (1)
      +- BroadcastExchange (9)
         +- Project (8)
            +- Filter (7)
               +- Scan csv  (6)


(1) Scan parquet 
Output [10]: [station_id#118, nom_station#119, code_arr#120, capacite#121, velos_meca#122, velos_elec#123, bornettes_libres#124, horodatage#125, annee#126, mois#127]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/data/velib/parquet]
PushedFilters: [IsNotNull(horodatage), IsNotNull(capacite), GreaterThan(capacite,0)]
ReadSchema: struct<station_id:int,nom_station:string,code_arr:int,capacite:int,velos_meca:int,velos_elec:int,bornettes_libres:int,horodatage:string>

(2) Filter
Input [10]: [station_id#118, nom_station#119, code_arr#120, capacite#121, velos_meca#

In [33]:
# [EXERCICE]
# Calculez, pour chaque combinaison (arrondissement, est_pluie),
# le taux_occupation moyen et le nombre de snapshots.
# Triez par arrondissement croissant, puis par est_pluie.
#
# Indice : utilisez groupBy(...).agg(F.mean(...).alias(...), F.count("*")...)
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :
# code_arr = code d'arrondissement (1-20 = Paris intra-muros, les autres = communes voisines).
# Comparer, pour un même arrondissement, la ligne est_pluie=false et la ligne est_pluie=true
# donne d'un coup d'œil l'effet de la pluie ; l'analyse statistique complète est au carnet 3.
(
    df_joint
    .groupBy("code_arr", "est_pluie")
    .agg(
        F.mean("taux_occupation").alias("taux_moyen"),
        F.count("*").alias("nb_snapshots"),
    )
    .orderBy("code_arr", "est_pluie")
    .show(40)
)

+--------+---------+-------------------+------------+
|code_arr|est_pluie|         taux_moyen|nb_snapshots|
+--------+---------+-------------------+------------+
|       1|    false| 0.5446380537590367|      183225|
|       1|     true| 0.5757651367531396|        8519|
|       2|    false|0.43177207235961595|      194252|
|       2|     true| 0.4357129829640348|        8981|
|       3|    false| 0.5725859525422418|      104809|
|       3|     true| 0.6218706368899916|        4836|
|       4|    false| 0.5563542409573974|      197703|
|       4|     true| 0.6106770249728557|        9210|
|       5|    false|0.47689798082086526|      277489|
|       5|     true|  0.486698127486932|       12817|
|       6|    false| 0.4767951200429157|      247953|
|       6|     true| 0.4927731515469334|       11442|
|       7|    false| 0.5894762279992507|      218913|
|       7|     true| 0.6125771570151591|       10292|
|       8|    false|0.34705372929021655|      392568|
|       8|     true|0.345575

---
## 2.7 Persistance en memoire : `.cache()` et `.persist()`

Nous allons utiliser `df_joint` intensivement pendant le reste du cours.
Mettons-le en cache pour eviter de recalculer la jointure a chaque action.


In [34]:
# .cache() == .persist(StorageLevel.MEMORY_AND_DISK)
# Si les donnees ne tiennent pas en RAM, Spark deborde sur disque.

t0 = time.perf_counter()
df_joint.cache()
df_joint.count()   # force la materialisation du cache
t_cache = time.perf_counter() - t0
print(f"Mise en cache (premiere passe) : {t_cache:.2f} s")

# Deuxieme passe -- depuis le cache
t0 = time.perf_counter()
df_joint.count()
t_hit = time.perf_counter() - t0
print(f"Lecture depuis le cache        : {t_hit:.2f} s")
print(f"Gain                           : x{t_cache/t_hit:.1f}")
print()
print("Allez dans Spark UI -> Storage pour voir la taille du cache.")


Mise en cache (premiere passe) : 8.50 s
Lecture depuis le cache        : 0.09 s
Gain                           : x96.8

Allez dans Spark UI -> Storage pour voir la taille du cache.


In [35]:
# StorageLevel disponibles (par ordre de rapidite / consommation memoire)
from pyspark import StorageLevel

print("StorageLevels disponibles :")
print("  MEMORY_ONLY          : RAM uniquement (eviction si plein)")
print("  MEMORY_AND_DISK      : RAM, puis disque si plein  [defaut de .cache()]")
print("  DISK_ONLY            : Disque uniquement (lent mais stable)")
print("  MEMORY_ONLY_SER      : RAM, serialise (moins de RAM, plus de CPU)")
print("  OFF_HEAP             : Memoire hors JVM (necessite configuration)")
print()
print("Regle pratique :")
print("  - Petits DataFrames reutilises souvent -> MEMORY_ONLY")
print("  - Gros DataFrames reutilises -> MEMORY_AND_DISK")
print("  - Ne pas cacher si utilise une seule fois -> overhead inutile")


StorageLevels disponibles :
  MEMORY_ONLY          : RAM uniquement (eviction si plein)
  MEMORY_AND_DISK      : RAM, puis disque si plein  [defaut de .cache()]
  DISK_ONLY            : Disque uniquement (lent mais stable)
  MEMORY_ONLY_SER      : RAM, serialise (moins de RAM, plus de CPU)
  OFF_HEAP             : Memoire hors JVM (necessite configuration)

Regle pratique :
  - Petits DataFrames reutilises souvent -> MEMORY_ONLY
  - Gros DataFrames reutilises -> MEMORY_AND_DISK
  - Ne pas cacher si utilise une seule fois -> overhead inutile


### Mesure du gain du cache sur la jointure répétée (contrainte n°6)

La cellule précédente compare la 1re passe (qui remplit le cache) et la 2e. Pour une mesure
plus rigoureuse, on répète **la même agrégation 3 fois** sur la jointure, **sans** puis **avec**
cache, et on relève les durées. Sans cache, Spark recalcule à chaque action toute la chaîne
(lecture Parquet -> nettoyage -> jointure) ; avec cache, il relit des blocs déjà en mémoire.

> Dans le Spark UI, onglet **Storage**, on voit la taille du DataFrame en cache ; onglet **SQL**,
> le plan sans cache contient `Scan parquet` et `BroadcastHashJoin`, celui avec cache un simple
> `InMemoryTableScan`.

In [36]:
# Dictionnaire partagé qui collecte toutes les mesures du projet (repris dans le bilan / README)
MESURES = {}

def chronometrer(fonction, repetitions: int = 3) -> list[float]:
    """Exécute `fonction` plusieurs fois et renvoie la durée de chaque exécution.

    Args:
        fonction: Callable sans argument qui déclenche une action Spark.
        repetitions: Nombre d'exécutions consécutives.

    Returns:
        Liste des durées, en secondes.
    """
    durees = []
    for _ in range(repetitions):
        t0 = time.perf_counter()
        fonction()
        durees.append(time.perf_counter() - t0)
    return durees

# Même calcul aval dans les deux cas : moyenne du taux par station et par heure
aval = lambda: df_joint.groupBy("station_id", "heure").agg(F.avg("taux_occupation")).count()

# 1) SANS cache : on retire d'abord le cache posé plus haut
df_joint.unpersist(blocking=True)
t_sans = chronometrer(aval)

# 2) AVEC cache : la 1re passe remplit le cache (on la mesure à part), les suivantes le lisent
df_joint.cache()
t_remplissage = chronometrer(aval, repetitions=1)[0]
t_avec = chronometrer(aval)

print("Durées sans cache  :", [f"{t:.1f}s" for t in t_sans])
print(f"Remplissage du cache (1re passe) : {t_remplissage:.1f} s")
print("Durées avec cache  :", [f"{t:.1f}s" for t in t_avec])
gain = (sum(t_sans) / len(t_sans)) / (sum(t_avec) / len(t_avec))
print(f"Gain moyen par passe : x{gain:.1f}")

MESURES["cache_jointure"] = {"sans_cache_s": t_sans, "remplissage_s": t_remplissage,
                              "avec_cache_s": t_avec, "gain": gain}
# Mesure conservée dans un fichier JSON commun (reprise dans le bilan du carnet 6)
import mesures
mesures.enregistrer(OUTPUT_DIR / "mesures.json", "cache_jointure", MESURES["cache_jointure"])
# Lecture : le cache coûte une passe de remplissage et la rentabilise dès la 2e utilisation.
# Il vaut donc le coup quand on réutilise le DataFrame plusieurs fois, comme df_joint dans la
# suite du projet.

Durées sans cache  : ['3.3s', '3.4s', '3.2s']
Remplissage du cache (1re passe) : 8.0 s
Durées avec cache  : ['0.2s', '0.1s', '0.1s']
Gain moyen par passe : x20.1


---
## 2.8 Ecriture en Parquet partitionne

La table `df_joint` est la table centrale du projet. Nous allons la persister
sur disque en Parquet, partitionne par annee et par mois, pour que les
traitements des jours suivants n'aient pas a la reconstruire.


In [37]:
OUTPUT_VELIB_CONSOLIDE = OUTPUT_DIR / "disponibilite_consolidee.parquet"

# Selection finale des colonnes (on elimine les colonnes intermediaires)
colonnes_finales = [
    "station_id", "nom_station", "code_arr", "capacite",
    "horodatage", "velos_meca", "velos_elec", "bornettes_libres",
    "taux_occupation", "statut",
    "annee", "mois", "jour_sem", "heure", "est_weekend",
    "temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm", "est_pluie"
]

df_sortie = df_joint.select(*colonnes_finales)

t0 = time.perf_counter()
(
    # Ecrire au format Parquet en réécrivant sur les données existantesn et ne parittionnant sur les années et les mois
    df_sortie.write
    .mode("overwrite")               # remplace l'existant : le carnet est rejouable à volonté
    .partitionBy("annee", "mois")    # un dossier par mois -> lectures filtrées bien plus rapides
    .parquet(str(OUTPUT_VELIB_CONSOLIDE))
)
t_write = time.perf_counter() - t0
print(f"Ecriture en {t_write:.1f} s")

# Verification : lecture selective d'une partition
df_verif = spark.read.parquet(str(OUTPUT_VELIB_CONSOLIDE)).filter("annee = 2021 AND mois = 1")
print(f"Janvier 2021 : {df_verif.count():,} lignes")
print(f"Schema : {df_verif.columns}")

Ecriture en 8.2 s
Janvier 2021 : 1,486,486 lignes
Schema : ['station_id', 'nom_station', 'code_arr', 'capacite', 'horodatage', 'velos_meca', 'velos_elec', 'bornettes_libres', 'taux_occupation', 'statut', 'jour_sem', 'heure', 'est_weekend', 'temperature_c', 'humidite_pct', 'vent_kmh', 'precipitation_mm', 'est_pluie', 'annee', 'mois']


In [38]:
# Comparaison de la taille CSV vs Parquet
import os

def taille_dossier_mb(path: Path) -> float:
    """
    Calcule la taille d'un dossier en mégaoctets.

    Args:
        path: Dossier à mesurer (parcouru récursivement) ou fichier isolé.

    Returns:
        Somme des tailles de tous les fichiers, en Mo (1 Mo = 1 048 576 octets).

    Example:
        >>> taille_dossier_mb(Path("../data/velib/raw")) > 0
        True
    """
    # rglob("*") descend dans les sous-dossiers annee=…/mois=… créés par partitionBy
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1_048_576

t_csv   = taille_dossier_mb(VELIB_RAW_DIR)
t_parq  = taille_dossier_mb(OUTPUT_VELIB_CONSOLIDE)
ratio   = t_csv / t_parq if t_parq > 0 else 0

print(f"Taille CSV bruts (compresses .gz) : {t_csv:.1f} MB")
print(f"Taille Parquet consolide          : {t_parq:.1f} MB")
print(f"Rapport CSV/Parquet               : x{ratio:.1f}")
print()
print("Le Parquet est plus petit car :")
print("  1. Stockage colonnaire : on compresse des valeurs homogenes")
print("  2. Encodage par dictionnaire pour les strings repetitives (noms de stations)")
print("  3. Run-Length Encoding sur les colonnes de partition (annee, mois)")

Taille CSV bruts (compresses .gz) : 195.3 MB
Taille Parquet consolide          : 71.1 MB
Rapport CSV/Parquet               : x2.7

Le Parquet est plus petit car :
  1. Stockage colonnaire : on compresse des valeurs homogenes
  2. Encodage par dictionnaire pour les strings repetitives (noms de stations)
  3. Run-Length Encoding sur les colonnes de partition (annee, mois)


---
## Bilan du Jour 1

### Ce que nous avons fait

| Etape | API | Concept cle |
|-------|-----|-------------|
| Chargement CSV brut | RDD | `sc.textFile()`, parsing manuel |
| Filtrage et transformation | RDD | `map()`, `filter()`, evaluation paresseuse |
| Agregation par cle | RDD | `reduceByKey()` vs `groupByKey()` |
| Profil horaire | RDD | `flatMap()`, calcul de moyenne distribuee |
| Jointure RDD | RDD | `join()` sur paires (cle, valeur) |
| Lecture Parquet | DataFrame | `spark.read.parquet()`, predicate pushdown |
| Exploration statistique | DataFrame | `describe()`, comptage de nulls |
| Nettoyage | DataFrame | `dropna()`, `when()`, `greatest()` |
| Features temporelles | DataFrame | `year()`, `hour()`, `dayofweek()` |
| Jointure broadcast | DataFrame | `broadcast()`, eviter le shuffle |
| Persistance | DataFrame/RDD | `.cache()`, `.persist()`, `StorageLevel` |
| Ecriture partitionnee | DataFrame | `write.partitionBy().parquet()` |

### Concepts fondamentaux a retenir

1. **Evaluation paresseuse** : les transformations construisent un plan, les actions l'executent.
2. **Le DAG** : lire le Spark UI est une competence essentielle, pas optionnelle.
3. **reduceByKey > groupByKey** : toujours combiner localement avant de shuffler.
4. **DataFrame > RDD** : l'optimiseur Catalyst rend les DataFrames significativement plus
   rapides que les RDD pour les memes operations.
5. **Cache strategique** : ne mettre en cache que ce qui est reutilise plusieurs fois.
6. **Broadcast join** : quand une table est petite, eviter le shuffle de la grosse table.

### Pour demain (Jour 2)

La table `disponibilite_consolidee.parquet` sera le point de depart du Jour 2.
Nous allons l'interroger avec Spark SQL (fenetrage temporel, requetes analytiques),
puis la connecter a un flux simule de mises a jour en temps reel avec Structured Streaming.


In [39]:
# Nettoyage de fin de session
# Toujours liberer la SparkSession proprement pour liberer les ressources
print("Arret de la SparkSession...")
spark.stop()
print("SparkSession arretee. Bonne nuit !")


Arret de la SparkSession...


SparkSession arretee. Bonne nuit !
